# Filament segmentation: final inference

Rebuilds the two final submissions from the 10 U-Net checkpoints (EfficientNet-B4, 5 folds each of
N1 = whole image @1024 and N2 = 1024 crops @2048) using only the test H-alpha images.

* **final 1**: N1 x5 + N2 x5 maps averaged 1:1 at 2048, hysteresis (t 0.4, seed 0.99, area 200),
  oriented fragment grouping (r 6, line 16), FP filter (`configs/fpfilter_ens_n1n2_orient.pkl`).
* **final 2**: N1 x5 maps upsampled to 2048, hysteresis (t 0.4, seed 0.99, area 200, r 12), no filter.

Every map is 8x TTA (4 rotations x flip), saved as uint8. On a 2-GPU machine N1 and N2 run in parallel.

In [ ]:
# parameters
import os, sys
from pathlib import Path
ON_KAGGLE = os.path.isdir('/kaggle/input')
REPO = Path('/kaggle/working/repo') if ON_KAGGLE else Path.cwd()
WEIGHTS = Path(os.environ.get('FILSEG_WEIGHTS', REPO / 'weights'))
OUT = Path('/kaggle/working') if ON_KAGGLE else REPO / 'outputs' / 'final'
N1 = [f'effb4_s1024_f{k}' for k in range(5)]
N2 = [f'effb4_c2048_f{k}' for k in range(5)]

In [ ]:
import glob, subprocess
os.chdir(REPO)
if ON_KAGGLE:
    imgs = glob.glob('/kaggle/input/**/test_images', recursive=True)
    assert imgs, 'attach the competition data'
    data = REPO / 'data'
    data.mkdir(exist_ok=True)
    link = data / 'MAGFiLO_1.0_Kaggle_2026'
    if not link.exists():
        link.mkdir()
        (link / 'test').mkdir()
        (link / 'test' / 'test_images').symlink_to(imgs[0])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch==0.5.0'], check=True)
import torch, segmentation_models_pytorch as smp
print('torch', torch.__version__, 'smp', smp.__version__, 'gpus', torch.cuda.device_count())
missing = [n for n in N1 + N2 if not (WEIGHTS / f'{n}.pt').exists()]
assert not missing, f'missing checkpoints in {WEIGHTS}: {missing}'

In [ ]:
# 8x-TTA test maps per checkpoint; N1 on GPU 0, N2 on GPU 1 when there are two
import time
t0 = time.time()
ngpu = max(torch.cuda.device_count(), 1)
def chain(names, gpu):
    cmds = [f'{sys.executable} scripts/infer_probs.py --ckpt {WEIGHTS}/{n}.pt --which test --tta '
            f'--out {OUT}/maps/{n}' for n in names]
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu % ngpu))
    return subprocess.Popen(' && '.join(cmds), shell=True, env=env)
procs = [chain(N1, 0), chain(N2, 1)] if ngpu > 1 else [chain(N1 + N2, 0)]
assert all(p.wait() == 0 for p in procs), 'inference failed'
print(f'maps done in {(time.time() - t0) / 60:.1f} min')

In [ ]:
def run(cmd):
    print('>>', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)
m = OUT / 'maps'
run(f'{sys.executable} scripts/ensemble_probs.py --inputs ' + ' '.join(str(m / n) for n in N1) + f' --out {m}/n1')
run(f'{sys.executable} scripts/ensemble_probs.py --inputs ' + ' '.join(str(m / n) for n in N2) + f' --out {m}/n2')
run(f'{sys.executable} scripts/ensemble_probs.py --inputs {m}/n1 {m}/n2 --out {m}/n1n2')
run(f'{sys.executable} scripts/ensemble_probs.py --inputs {m}/n1 {m}/n2 --weights 1 0 --out {m}/n1up')
run(f'{sys.executable} scripts/make_submission.py --probs {m}/n1n2 --filter configs/fpfilter_ens_n1n2_orient.pkl '
    f'--out {OUT}/submission_final1.csv')
run(f'{sys.executable} scripts/make_submission.py --probs {m}/n1up --post configs/post_n1up_hyst.json '
    f'--out {OUT}/submission_final2.csv')

In [ ]:
# hashes for the reproducibility check, then drop the maps (several GB)
import hashlib, shutil
dirs = ' '.join(f'{n}={m / n}' for n in N1 + N2 + ['n1', 'n2', 'n1n2', 'n1up'])
run(f'{sys.executable} scripts/map_hashes.py --dirs {dirs} --out {OUT}/map_hashes.json')
for f in ('submission_final1.csv', 'submission_final2.csv'):
    print(f, hashlib.sha256((OUT / f).read_bytes()).hexdigest())
if ON_KAGGLE:
    shutil.copy(OUT / 'submission_final1.csv', OUT / 'submission.csv')
    shutil.rmtree(m)
    shutil.rmtree(REPO)